In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression

PROJECT_ROOT = Path.cwd().parent  
path = PROJECT_ROOT / "data" / "interim" / "hourly_uniform.parquet"

df = pd.read_parquet(path)

# keep PM2.5 only (adjust string if needed)
df["pollutant_norm"] = (
    df["pollutant"].astype(str)
    .str.upper().str.replace(" ", "").str.replace(".", "")
)
df_pm25 = df[df["pollutant_norm"] == "PM25"].copy()

# ensure datetime and site_name exist
df_pm25["timestamp_local"] = pd.to_datetime(df_pm25["timestamp_local"])

# daily mean per site
daily = (
    df_pm25
    .set_index("timestamp_local")
    .groupby("site_name")["value"]
    .resample("D")
    .mean()
    .reset_index()
    .rename(columns={"value": "pm25_daily"})
)

# add year for later grouping
daily["year"] = daily["timestamp_local"].dt.year

daily.head()

,site_name,timestamp_local,pm25_daily,year
0,Alphington,2019-01-01 00:00:00+11:00,6.275000,2019
1,Alphington,2019-01-02 00:00:00+11:00,5.120833,2019
2,Alphington,2019-01-03 00:00:00+11:00,7.908333,2019
3,Alphington,2019-01-04 00:00:00+11:00,13.258333,2019
4,Alphington,2019-01-05 00:00:00+11:00,7.354167,2019


In [7]:
threshold = 15  # adjust if you want a different guideline

summary_site_year = (
    daily
    .assign(exceed = daily["pm25_daily"] > threshold)
    .groupby(["site_name", "year"])
    .agg(
        days=("pm25_daily", "count"),
        mean_pm25=("pm25_daily", "mean"),
        median_pm25=("pm25_daily", "median"),
        p95_pm25=("pm25_daily", lambda s: s.quantile(0.95)),
        exceed_days=("exceed", "sum"),
    )
    .reset_index()
)

summary_site_year["exceed_pct"] = (
    summary_site_year["exceed_days"] / summary_site_year["days"] * 100
)

summary_site_year.head()


,site_name,year,days,mean_pm25,median_pm25,p95_pm25,exceed_days,exceed_pct
0,Alphington,2019,253,7.745353,6.630000,16.137500,16,6.324111
1,Alphington,2020,293,8.400697,7.012500,20.065833,27,9.215017
2,Alphington,2021,1,16.868000,16.868000,16.868000,1,100.000000
3,Alphington,2022,353,7.347738,6.471783,14.829295,18,5.099150
4,Alphington,2023,364,6.654882,5.934188,12.447056,9,2.472527


In [2]:
# make sure year column exists
daily["year"] = daily["timestamp_local"].dt.year

print("Date range:", daily["timestamp_local"].min(), "->", daily["timestamp_local"].max())
print("Number of sites:", daily["site_name"].nunique())

# how many days of data per site per year?
coverage = (
    daily
    .groupby(["site_name", "year"])["pm25_daily"]
    .count()
    .reset_index(name="n_days")
)

# pivot for easy viewing: rows = sites, columns = years, values = number of days
coverage_pivot = coverage.pivot(index="site_name", columns="year", values="n_days")

# show the first few rows
coverage_pivot.head()

Date range: 2019-01-01 00:00:00+11:00 -> 2024-12-31 00:00:00+11:00
Number of sites: 17


year,2019,2020,2021,2022,2023,2024
site_name,,,,,,
Alphington,253.0,293.0,1.0,353.0,364.0,325.0
Bendigo,NaN,NaN,NaN,299.0,276.0,NaN
Box Hill,NaN,NaN,NaN,311.0,362.0,283.0
Brighton,NaN,NaN,NaN,355.0,356.0,363.0
Campbellfield,NaN,NaN,NaN,75.0,NaN,NaN


In [8]:
# Defining a simple rule for what counts as "enough" data
# in here, we choose a minimum number of days to treat a year as "usable"
MIN_DAYS = 250

coverage["usable"] = coverage["n_days"] >= MIN_DAYS

# quick summary: how many usable years per site?
usable_years_per_site = (
    coverage[coverage["usable"]]
    .groupby("site_name")["year"]
    .count()
    .sort_values(ascending=False)
)

usable_years_per_site.head(15)

site_name
Alphington       5
Traralgon        5
Melbourne CBD    5
Geelong South    4
Footscray        3
Box Hill         3
Brighton         3
Morwell South    3
Dandenong        2
Bendigo          2
Churchill        2
Moe              2
Melton           2
Morwell East     1
Newborough       1
Name: year, dtype: int64

In [11]:
# We filter daily data to only the long-history sites
usable_sites = ["Alphington", "Traralgon", "Melbourne CBD", "Geelong South"]

daily_use = daily[daily["site_name"].isin(usable_sites)].copy()

In [12]:
# Now, we compute annual PM2.5 per site
annual = (daily_use
          .groupby(["site_name", "year"])["pm25_daily"]
          .mean()
          .reset_index(name="pm25_annual_mean")
)

annual.head()   

,site_name,year,pm25_annual_mean
0,Alphington,2019,7.745353
1,Alphington,2020,8.400697
2,Alphington,2021,16.868000
3,Alphington,2022,7.347738
4,Alphington,2023,6.654882


In [14]:
# Now, we compute trends per site using a linear regression slope.
slopes = []

for site in usable_sites:
    sub = annual[annual["site_name"] == site][["year", "pm25_annual_mean"]]

    # we drop years with missing values
    sub = sub.dropna(subset=["pm25_annual_mean"]).copy()

    # If fewer than 2 years remain, skip (this means we cannot complete the slope)
    if len(sub) < 2:
        print(f"[skip] Not enough years for site: {site}")
        continue
    
    X = sub["year"].values.reshape(-1,1)
    y = sub["pm25_annual_mean"].values

    model = LinearRegression().fit(X,y)
    slopes.append({
        "site": site,
        "slope_pm25_per_year": model.coef_[0],
        "intercept": model.intercept_,
        "n_years": len(sub)
    })

pd.DataFrame(slopes)

,site,slope_pm25_per_year,intercept,n_years
0,Alphington,-0.356440,729.746553,6
1,Traralgon,-0.657108,1337.077912,6
2,Melbourne CBD,0.082229,-158.281970,5
3,Geelong South,-0.557883,1134.415589,6
